In [ ]:
from datetime import datetime, timedelta
from dotenv import load_dotenv
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from matplotlib.pyplot import subplots
from pandas import DataFrame, concat
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from foundation.entities.stock import Stock
load_dotenv("../../.env")

mag7_tech = ["AAPL","NVDA","META","INTC","AMZN","GOOGL","MSFT"]
mag7_health = ["LLY","NVO","VRTX","ABBV","ISRG","HCA", "UNH", "HIMS"]
mag7_financial = ["V","MA","PYPL","AXP","BLK"]
other = ["INTC","AVGO","RDDT","UBER","SNOW"]
df = DataFrame()
for symbol in mag7_tech + mag7_health + mag7_financial + other:
    stock = Stock(symbol=symbol, company=symbol)
    stock.populate_ratings()
    _df = stock.ratings.to_frame(); _df["stk"] = stock.symbol
    df = concat([df, _df])
cutoff = (datetime.now() - timedelta(days=60)).date()
df = df[df["date"] >= cutoff]
df;

In [ ]:
fig, ax = subplots(nrows=2,ncols=1,figsize=(18,6)); #fig.subplots_adjust(hspace=0.25)
df_melted = df.melt(id_vars=["date", "stk", "source"], value_vars=["buy", "hold", "sell"], var_name="rating", value_name="score")
sources = df_melted["source"].unique()
for idx, src in enumerate(sources):
    sns.barplot(data=df_melted[df_melted["source"] == src], x="stk", y="score", hue="rating", ax=ax[idx])
    ax[idx].set_title(src)
fig.tight_layout()

In [ ]:
# clustering/options_clustering.py


def load_data(csv_path: str) -> pd.DataFrame:
    """Load options features dataset."""
    df = pd.read_csv(csv_path, parse_dates=["date"])
    return df


def preprocess_data(df: pd.DataFrame) -> pd.DataFrame:
    """Select and scale numerical features for clustering."""
    feature_cols = [
        "put_call_volume_ratio",
        "put_call_oi_ratio",
        "avg_iv_calls",
        "avg_iv_puts",
        "iv_skew",
        "vwap_strike_calls",
        "vwap_strike_puts",
        "strike_bias",
    ]
    X = df[feature_cols].copy()
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    return X_scaled


def cluster_data(X_scaled, n_clusters=3, random_state=42):
    """Apply KMeans clustering."""
    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    return labels, kmeans


def reduce_dimensions(X_scaled, n_components=2):
    """Apply PCA for visualization."""
    pca = PCA(n_components=n_components, random_state=42)
    X_pca = pca.fit_transform(X_scaled)
    return X_pca


def plot_clusters(X_pca, labels):
    """Plot clusters in 2D PCA space."""
    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap="viridis", alpha=0.7)
    plt.xlabel("PCA Component 1")
    plt.ylabel("PCA Component 2")
    plt.title("Options Data Clusters (KMeans)")
    plt.colorbar(scatter, label="Cluster")
    plt.show()


csv_path = "../../data/options_features.csv"  # replace with your actual path
df = load_data(csv_path)
X_scaled = preprocess_data(df)
labels, kmeans = cluster_data(X_scaled, n_clusters=3)
X_pca = reduce_dimensions(X_scaled)
plot_clusters(X_pca, labels)

# attach labels back to df
df["cluster"] = labels
print(df[["date", "ticker", "spot","cluster"]])